# rank-world-size-args — worked example 1: All-reduce protocol: every rank sends and receives

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank-world-size-args`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In an all-reduce operation, every rank must both send its local tensor to every other rank and receive tensors from every other rank. Unlike a broadcast (one sender, all receivers) or reduce (all senders, one receiver), all-reduce is symmetric: every rank acts as both sender and receiver. The standard signature `all_reduce(tensor, rank, world_size)` always includes `rank` and `world_size` so each spawned process knows its role and the total participant count.

## Worked solution

**Step 1 — Signature.** `all_reduce_protocol(tensor, rank, world_size)` follows the convention: `tensor` is the local data, `rank` is this process's index (0-based), and `world_size` is the total number of processes.

**Step 2 — Every rank sends.** For each `other` in `range(world_size)` where `other != rank`, this rank sends to `other`. The list comprehension produces these in ascending order.

**Step 3 — Every rank receives.** For each `other` in `range(world_size)` where `other != rank`, this rank receives from `other`.

**Step 4 — Return combined action list.** Sends come first (ascending by target), then receives (ascending by source). This describes the full communication pattern for one rank in an all-reduce.

In [ ]:
import torch as t

def all_reduce_protocol(tensor, rank: int, world_size: int) -> list:
    """Return the (action, other_rank) list describing what this rank does in all-reduce."""
    sends = [('send', other) for other in range(world_size) if other != rank]
    recvs = [('recv', other) for other in range(world_size) if other != rank]
    return sends + recvs

# Verify for world_size=3, rank=1
world_size = 3
for rank in range(world_size):
    actions = all_reduce_protocol(None, rank, world_size)
    print(f'rank={rank}: {actions}')

# Rank 1 should send to 0 and 2, then recv from 0 and 2
rank1 = all_reduce_protocol(None, 1, 3)
print()
print(f'Rank 1 sends: {[a for a in rank1 if a[0]=="send"]}')
print(f'Rank 1 recvs: {[a for a in rank1 if a[0]=="recv"]}')